In [ ]:
# ============================================================
# CELL 1 — Installs, imports, prompt, config
# ============================================================

!pip install -U google-genai openpyxl pandas -q

from google import genai
from google.genai import types
from google.colab import userdata
import pandas as pd
import json, time

client     = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))
GEMINI_MODEL = "gemini-2.5-flash-lite"

SYSTEM_PROMPT = (
    "You are a precise and conservative text editor."
    "Your task is to paraphrase  claims while strictly preserving their original meaning, factual content, and verifiability conditions."
)

USER_TEMPLATE = """Rewrite the following claim in different words.

Guidelines:
- Preserve all numbers, dates, statistics, and factual assertions exactly.
- Preserve the original logical direction and strength of the claim (do not reverse, weaken, or hedge it).
- Maintain all information necessary to verify the claim. Do not remove or generalize named entities if this would affect interpretability.
- Keep approximately the same level of specificity and length as the original claim.
- Use a substantially different sentence structure from the original (e.g., change active to passive voice, reorder subject/predicate, or alter the syntactic construction), while keeping the same underlying proposition.
- If the claim is very short, ensure at least three lexical substitutions are made in addition to structural variation.

Before writing, internally ensure that (not include this check in your output):
- the paraphrase expresses the same testable claim,
- no information relevant for fact-checking has been lost or altered

Output:
- A single rewritten claim.
- No preamble, no explanation.

Claim: {claim}"""

print("✅ Cell 1 OK")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 806.3/806.3 kB 35.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 103.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.6/245.6 kB 17.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.47.0, but you have google-auth 2.52.0 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.3 which is incompatible.
google-cloud-aiplatform 1.148.1 requires google-genai<2.0.0,>=1.66.0; python_version >= "3.10", but you have google-genai 2.2.0 which is incompatible.
db-dtypes 1.5.1 requires pandas<3.0.0,>=1.5.3, but you have pandas 3.0.3 whi

In [ ]:
# ============================================================
# CELL 2 — Load data + build JSONL + upload
# ============================================================

df = pd.read_excel("claims_for_api.xlsx")
df.columns = [c.strip() for c in df.columns]

# Scegli la parte da lanciare (cambia ogni volta)          #changeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeee
#df = df.iloc[:3200]       # Part 1
# df = df.iloc[3200:6400]  # Part 2
# df = df.iloc[6400:9600]  # Part 3
df = df.iloc[9600:]      # Part 4

# Build JSONL
jsonl_path = "gemini_paraphrase_input.jsonl"
with open(jsonl_path, "w", encoding="utf-8") as f:
    for idx, row in df.iterrows():
        claim = str(row[claim_col]).strip()
        task = {
            "key": f"row_{idx}",
            "request": {
                "systemInstruction": {        # camelCase obbligatorio
                    "parts": [{"text": SYSTEM_PROMPT}]
                },
                "contents": [{
                    "role": "user",
                    "parts": [{"text": USER_TEMPLATE.format(claim=claim)}]
                }],
                "generationConfig": {         # camelCase obbligatorio
                    "temperature": 0.7,
                    "maxOutputTokens": 2048
                }
            }
        }
        f.write(json.dumps(task, ensure_ascii=False) + "\n")

print(f"✅ JSONL scritto: {jsonl_path}")

# Upload — mime_type='text/plain' (application/jsonl ha un bug noto nell'SDK)
print("Uploading file...")
uploaded_file = client.files.upload(
    file=jsonl_path,
    config=types.UploadFileConfig(
        display_name="gemini-batch-justifications",
        mime_type="text/plain"
    )
)
print(f"✅ Upload OK")
print(f"   Name: {uploaded_file.name}")
print(f"   URI:  {uploaded_file.uri}")

✅ JSONL scritto: gemini_paraphrase_input.jsonl
Uploading file...
✅ Upload OK
   Name: files/qrzg7f6lnqsq
   URI:  https://generativelanguage.googleapis.com/v1beta/files/qrzg7f6lnqsq


In [ ]:
# ============================================================
# CELL 3 — Submit batch job
# ============================================================

print("Submitting batch job...")
batch_job = client.batches.create(
    model=GEMINI_MODEL,
    src=uploaded_file.name,       # .name non .uri ← era il bug principale
    config={"display_name": "justification-batch"}
)

GEMINI_BATCH_NAME = batch_job.name
print(f"🚀 Job creato!")
print(f"   Name:  {GEMINI_BATCH_NAME}")
print(f"   State: {batch_job.state}")

Submitting batch job...
🚀 Job creato!
   Name:  batches/x2t1zbj24kp594728eov6l6laq4yz1jhqnwp
   State: JobState.JOB_STATE_PENDING


In [1]:
# ============================================================
# CELL 4 — Polling fino a completamento
# ============================================================
from google import genai
from google.genai import types
from google.colab import userdata
import pandas as pd
import json, time

client     = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))


# Se Colab si disconnette, incolla qui il batch name:
GEMINI_BATCH_NAME = "batches/x2t1zbj24kp594728eov6l6laq4yz1jhqnwp"        #CHANGEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEE !!!!!!!!!!!!!!!!!!!!!!!!!!!!!

TERMINAL_STATES = {"JOB_STATE_SUCCEEDED", "JOB_STATE_FAILED", "JOB_STATE_CANCELLED"}

print(f"Polling: {GEMINI_BATCH_NAME}\n")
while True:
    job   = client.batches.get(name=GEMINI_BATCH_NAME)
    state = job.state.name
    print(f"  [{time.strftime('%H:%M:%S')}] {state}")

    if state in TERMINAL_STATES:
        break
    time.sleep(30)

if state == "JOB_STATE_SUCCEEDED":
    print("\n✅ Completato!")
else:
    raise RuntimeError(f"Job terminato con stato: {state}\nErrore: {getattr(job, 'error', 'N/A')}")

Polling: batches/x2t1zbj24kp594728eov6l6laq4yz1jhqnwp

  [13:35:49] JOB_STATE_SUCCEEDED

✅ Completato!


In [2]:
# @title
# ============================================================
# CELL 5 — Download + parse + salva Excel
# ============================================================

from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

df = pd.read_excel("claims_for_api.xlsx")
df.columns = [c.strip() for c in df.columns]
claim_col = "statement"
print(f"Rows loaded: {len(df)}")

# Download risultati
result_bytes = client.files.download(file=job.dest.file_name)
result_text  = result_bytes.decode("utf-8")

# Parse
results = []
for line in result_text.splitlines():
    if not line.strip():
        continue
    obj = json.loads(line)
    key = obj.get("key", "")
    try:
        text   = obj["response"]["candidates"][0]["content"]["parts"][0]["text"].strip()
        status = "success"
    except (KeyError, IndexError) as e:
        text   = str(obj.get("error", e))
        status = "error"
    results.append({"key": key, "status": status, "paraphrase": text})

results_df = pd.DataFrame(results)
print(f"Successi: {(results_df['status']=='success').sum()}/{len(results_df)}")
print(f"Errori:   {(results_df['status']=='error').sum()}/{len(results_df)}")

# Merge con df originale
df_out = df.copy()
df_out["key"] = [f"row_{i}" for i in df.index]
df_out = df_out.merge(results_df[["key", "paraphrase"]], on="key", how="left")
df_out = df_out.drop(columns=["key"])
df_out = df_out.rename(columns={claim_col: "Claim"})

# Salva Excel
output_path = "gemini_paraphrases_part4.xlsx"   # CHANGEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEE
df_out[["Claim", "paraphrase"]].to_excel(output_path, index=False, sheet_name="Results")

wb = load_workbook(output_path)
ws = wb["Results"]

header_fill = PatternFill("solid", fgColor="1A5276")
header_font = Font(bold=True, color="FFFFFF", name="Arial", size=11)
cell_font   = Font(name="Arial", size=10)
wrap_align  = Alignment(wrap_text=True, vertical="top")

col_widths = [70, 80]
for col_idx, (cell, width) in enumerate(zip(ws[1], col_widths), start=1):
    cell.fill      = header_fill
    cell.font      = header_font
    cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    ws.column_dimensions[get_column_letter(col_idx)].width = width

ws.row_dimensions[1].height = 30
for row in ws.iter_rows(min_row=2):
    for cell in row:
        cell.font      = cell_font
        cell.alignment = wrap_align

ws.freeze_panes = "A2"
wb.save(output_path)
print(f"\n✅ Salvato: {output_path}")

for i, row in df_out.head(3).iterrows():
    print(f"\n─── Row {i} ───")
    print(f"CLAIM:     {str(row['Claim'])[:100]}")
    print(f"PARAPHRASE: {str(row['paraphrase'])[:200]}")

Rows loaded: 12791
Successi: 3190/3191
Errori:   1/3191

✅ Salvato: gemini_paraphrases_part4.xlsx

─── Row 0 ───
CLAIM:     Says the Annies List political group supports third-trimester abortions on demand.
PARAPHRASE: nan

─── Row 1 ───
CLAIM:     When did the decline of coal start? It started when natural gas took off that started to begin in (P
PARAPHRASE: nan

─── Row 2 ───
CLAIM:     Hillary Clinton agrees with John McCain "by voting to give George Bush the benefit of the doubt on I
PARAPHRASE: nan
